In [ ]:
import requests
from flask import Flask, request, jsonify
import re

app = Flask(__name__)

# 네이버 쇼핑 API 인증 정보
CLIENT_ID = ""  # 네이버 API Client ID
CLIENT_SECRET = ""  # 네이버 API Client Secret

# HTML 태그 제거 함수
def remove_html_tags(text):
    return re.sub(r'<[^>]*>', '', text)

# 네이버 쇼핑 API를 통해 상품 검색
def search_product(product_name):
    url = "https://openapi.naver.com/v1/search/shop.json"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET
    }
    params = {"query": product_name, "display": 1, "start": 1, "sort": "sim"}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)
        if response.status_code == 200:
            result = response.json()
            if result['items']:
                item = result['items'][0]
                return {
                    "title": remove_html_tags(item['title']),
                    "category1": item['category1'],
                    "category2": item['category2'],
                    "category3": item['category3'],
                    "category4": item['category4'],
                    "price": item['lprice'],
                    "link": item['link']
                }
        else:
            print(f"❌ [ERROR] 네이버 API 요청 실패 - 상태 코드: {response.status_code}")
            return None
    except requests.Timeout:
        print(f"⏰ [WARNING] '{product_name}' 요청 시간 초과")
        return None
    except requests.RequestException as e:
        print(f"❌ [ERROR] 요청 실패: {e}")
        return None

# 1차 카테고리 기준으로 식품 여부 판별
def is_food(product_info):
    food_categories = ["식품", "음료", "건강식품", "농산물", "수산물", "축산물", "가공식품", "신선식품"]
    category1 = product_info.get('category1', 'Unknown')
    return category1 in food_categories

# OCR로 받은 재료명을 식품 여부에 따라 분류하는 API 엔드포인트
@app.route('/ocr', methods=['POST'])
def process_ocr_data():
    try:
        print("\n📥 [INFO] 요청 수신: /ocr")

        if not request.is_json:
            return jsonify({"status": "error", "message": "Invalid JSON format"}), 400

        ocr_data = request.get_json()
        product_names = ocr_data.get("products", [])

        if not product_names:
            return jsonify({"status": "error", "message": "No products received"}), 400

        food_results = []

        # 각 제품명에 대해 네이버 API 검색 → 식품 여부 판별 → 결과 저장
        for product_name in product_names:
            product_info = search_product(product_name)
            if product_info and is_food(product_info):
                food_results.append({
                    "original_name": product_name,  # OCR에서 인식한 원본 이름
                    "title": product_info["title"],
                    "category1": product_info["category1"],
                    "category2": product_info["category2"],
                    "category3": product_info["category3"],
                    "category4": product_info["category4"],
                    "price": product_info["price"],
                    "link": product_info["link"]
                })

        return jsonify({"status": "success", "classified_products": food_results}), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

# Flask 서버 실행
if __name__ == '__main__':
    print("\n🚀 [INFO] Flask 서버 시작...")
    app.run(host='0.0.0.0', port=5000, debug=True, use_reloader=False)
